In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
load_dotenv()
import operator

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

In [ ]:
class evaluation_schema(BaseModel):
    feedback: str = Field(description="Feedback on the essay")
    score:int = Field(description="Score out of 10",ge=0,le=10)

In [ ]:
structured_model=model.with_structured_output(evaluation_schema)

In [ ]:
essay="""The impact of climate change on global agriculture is profound and multifaceted. Rising temperatures, changing precipitation patterns, and increased frequency of extreme weather events are affecting crop yields and food security worldwide. In this essay, we will explore the various ways in which climate change is influencing agriculture and discuss potential strategies for adaptation and mitigation."""

In [ ]:
prompt = f"Evaluate the following essay on a scale of 0 to 10 and provide feedback:\n\n{essay}"

In [ ]:
structured_model.invoke(prompt).score

In [ ]:
# define state graph
class UPSC_state(TypedDict):
    essay: str 
    language_feedback: str
    analytical_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    average_score: float

In [ ]:
def evaluate_language(state:UPSC_state):
    prompt= f"Evaluate the language quality of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    output=structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'language_score': output.score}


In [ ]:
def evaluate_analytical(state:UPSC_state):
    prompt= f"Evaluate the analytical depth of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    output=structured_model.invoke(prompt)
    return {'analytical_feedback': output.feedback, 'analytical_score': output.score}

In [ ]:
def evaluate_clarity(state:UPSC_state):
    prompt= f"Evaluate the clarity of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    output=structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'clarity_score': output.score}

In [ ]:
def final_evaluation(state:UPSC_state):
    # summarize feedback
    prompt = f"""Based on the following feedback and scores, provide an overall evaluation of the essay, including an overall score out of 10 and a summary of strengths and weaknesses:\n\n{state['language_feedback']}\n\n{state['analytical_feedback']}\n\n{state['clarity_feedback']}"""
    output=model.invoke(prompt)
    # calculate average score
    average_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback': output, 'average_score': average_score}


In [ ]:
# graph
graph =StateGraph(UPSC_state)


graph.add_node("evaluate_language",evaluate_language)
graph.add_node("evaluate_analytical",evaluate_analytical)
graph.add_node("evaluate_clarity",evaluate_clarity) 
graph.add_node("final_evaluation",final_evaluation)

In [ ]:
#edges
graph.add_edge(START,"evaluate_language")
graph.add_edge(START,"evaluate_analytical")
graph.add_edge(START,"evaluate_clarity")

graph.add_edge("evaluate_language","final_evaluation")
graph.add_edge("evaluate_analytical","final_evaluation")
graph.add_edge("evaluate_clarity","final_evaluation")

graph.add_edge("final_evaluation",END)

workflow=graph.compile()

In [ ]:
intial_state = {
    'essay':essay}

In [ ]:
workflow.invoke(intial_state)